In [62]:
import chess
import numpy as np

def board_to_array(board):
    # Create an 18 x 8 x 8 array
    x = np.zeros((19, 8, 8), dtype=np.uint8)

    # Layers 0-11: Piece positions
    # White: pawn, knight, bishop, rook, queen, king
    # Black: pawn, knight, bishop, rook, queen, king
    piece_layer = {
        (chess.WHITE, chess.PAWN): 0,
        (chess.WHITE, chess.KNIGHT): 1,
        (chess.WHITE, chess.BISHOP): 2,
        (chess.WHITE, chess.ROOK): 3,
        (chess.WHITE, chess.QUEEN): 4,
        (chess.WHITE, chess.KING): 5,
        (chess.BLACK, chess.PAWN): 6,
        (chess.BLACK, chess.KNIGHT): 7,
        (chess.BLACK, chess.BISHOP): 8,
        (chess.BLACK, chess.ROOK): 9,
        (chess.BLACK, chess.QUEEN): 10,
        (chess.BLACK, chess.KING): 11,
    }

    for square, piece in board.piece_map().items():
        row = 7 - chess.square_rank(square)
        col = chess.square_file(square)
        layer = piece_layer[(piece.color, piece.piece_type)]
        x[layer, row, col] = 1

    # Layers 12-15: Castling rights
    x[12, :, :] = int(board.has_kingside_castling_rights(chess.WHITE))
    x[13, :, :] = int(board.has_queenside_castling_rights(chess.WHITE))
    x[14, :, :] = int(board.has_kingside_castling_rights(chess.BLACK))
    x[15, :, :] = int(board.has_queenside_castling_rights(chess.BLACK))

    # Layer 16: Location of a pawn that can legally be
    # en passant
    if board.ep_square is not None:
        ep_square = board.ep_square
        row = 7 - chess.square_rank(ep_square)
        col = chess.square_file(ep_square)
        x[16, row, col] = 1

    # Layer 17: Side to move
    x[17, :, :] = int(board.turn == chess.WHITE)

    # Board shape
    x[18, :, :] = 1

    return x


In [63]:
def add_moves(moves):
    board = chess.Board()
    for move_san in moves:
        move = board.parse_san(move_san)
        board.push(move)

        x = board_to_array(board)
        X.append(x)

In [64]:
scores = []
X = []
with open("../../data/regressor_games.jsonl", encoding="utf-8") as games:
    for game in games:
        game = game[game.find("1."):]
        game = game.replace("!","").replace("?","").replace("#","").replace("] }","").replace("{ [%eval","").replace("..","")
        game = game.split()
        del game[::3]
        del game[-1]
        scores.extend(float(eval) for eval in game[1::2])
        moves = game[::2]
        add_moves(moves)

X = np.stack(X)
scores = np.stack(scores)

In [65]:
print(X.shape)
print(scores.shape)

(33443, 19, 8, 8)
(32943,)
